# AstroPredict — Notebook 00  
## SHARP Context Library Builder (Demonstration Support)

This notebook constructs a reusable library of historical SHARP magnetic feature sequences to support real-time–style system demonstrations in later notebooks.

Due to the 24–48 hour latency in publicly available solar magnetogram data, real-time SHARP inputs are not accessible. This notebook addresses that limitation by preparing a controlled set of historical sequences that are format-compatible with the trained models.

The context library produced here is used exclusively for demonstration purposes in Notebook-04 and Notebook-05.  
It does not perform prediction, forecasting, simulation, or model training.

The role of this notebook is strictly infrastructural.


## Load ML-Ready Dataset

This cell loads the previously prepared ML-ready SHARP dataset from Google Drive.

The dataset contains cleaned magnetic features, temporal features, and flare labels,
as produced in **Notebook 01 (Data Preparation Pipeline)**.

The loaded DataFrame will be used to extract fixed-length SHARP sequences
for building the context library in subsequent steps.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import joblib

DATA_PATH = "/content/drive/MyDrive/AstroPredict_Final/ml_ready_dataset.csv"
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

print(df.shape)


(1356208, 19)


## Feature and Parameter Definition

This cell defines the feature columns, target label, and sequence length
used for extracting SHARP time-series windows.

- `FEATURE_COLS` specifies the same 13 magnetic and temporal features
  used during model training.
- `LABEL_COL` identifies the flare prediction target (24-hour horizon).
- `LOOKBACK` defines the fixed sequence length (30 timesteps) used
  for sequence construction.


In [ ]:
FEATURE_COLS = [
    "USFLUX","TOTUSJH","TOTUSJZ","TOTPOT",
    "R_VALUE","SAVNCPP","ABSNJZH","MEANALP",
    "R_VALUE_diff6h","TOTUSJH_diff6h","TOTPOT_diff6h",
    "R_VALUE_std6h","TOTUSJH_std6h"
]

LABEL_COL = "flare_label_24h"
LOOKBACK = 30


## Sequence Extraction by Activity Class

This cell constructs fixed-length SHARP time-series windows and separates
them into two groups based on the associated flare label.

For each active region (`harpnum`):
- Observations are sorted chronologically.
- Sliding windows of length `LOOKBACK` (30 timesteps) are extracted.
- Each window is assigned to:
  - **QUIET** if the corresponding label is 0
  - **ACTIVE** if the corresponding label is 1

The function returns two collections:
- `quiet_windows`: sequences not followed by a flare within 24 hours
- `active_windows`: sequences followed by a flare within 24 hours


In [ ]:
def extract_windows(df, feature_cols, label_col, lookback):
    quiet, active = [], []

    for _, g in df.groupby("harpnum"):
        g = g.sort_values("timestamp")
        X = g[feature_cols].values
        y = g[label_col].values

        for i in range(lookback, len(g)):
            window = X[i-lookback:i]
            label = y[i]

            if label == 0:
                quiet.append(window)
            else:
                active.append(window)

    return quiet, active

quiet_windows, active_windows = extract_windows(
    df, FEATURE_COLS, LABEL_COL, LOOKBACK
)

print("Quiet windows:", len(quiet_windows))
print("Active windows:", len(active_windows))


Quiet windows: 572072
Active windows: 774866


## Context Window Sampling

This cell selects a fixed number of representative SHARP time-series windows
from the previously extracted QUIET and ACTIVE pools.

- A random seed is set to ensure reproducibility.
- 300 windows are sampled from each class without replacement.
- The sampled windows are stored in a dictionary structure for later retrieval.

The resulting `context_library` contains:
- `QUIET`: 300 SHARP sequences labeled as non-flaring
- `ACTIVE`: 300 SHARP sequences labeled as flaring


In [ ]:
np.random.seed(42)

quiet_sample = np.random.choice(len(quiet_windows), 300, replace=False)
active_sample = np.random.choice(len(active_windows), 300, replace=False)

context_library = {
    "QUIET": [quiet_windows[i] for i in quiet_sample],
    "ACTIVE": [active_windows[i] for i in active_sample]
}


## Save Context Library

This cell serializes the constructed SHARP context library to persistent storage.

- The library is saved as a binary file using `joblib`.
- The output file contains pre-sampled SHARP time-series windows.
- This file is intended for reuse in later notebooks without recomputation.

The saved artifact enables consistent and repeatable loading of
context windows for real-time inference demonstrations.


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/AstroPredict_Final/sharp_context.pkl"
joblib.dump(context_library, SAVE_PATH)

print("✅ SHARP Context Library saved:", SAVE_PATH)


✅ SHARP Context Library saved: /content/drive/MyDrive/AstroPredict_Final/sharp_context.pkl


## Notebook Summary

This notebook prepares a reusable SHARP context library to support
real-time demonstration workflows in later stages of the AstroPredict project.

Due to the known latency in availability of SHARP magnetogram data,
direct real-time magnetic field measurements are not accessible.
This notebook addresses that limitation by organizing historical
SHARP time-series windows into a compact, structured library.

### What This Notebook Does
- Loads the ML-ready SHARP dataset produced in Notebook 01
- Extracts fixed-length (30-timestep) magnetic feature windows
- Separates windows into QUIET and ACTIVE categories based on flare labels
- Randomly samples representative windows from each category
- Saves the resulting context library as a reusable artifact


### Output Artifact
- `sharp_context.pkl`  
  A compact collection of representative SHARP time-series windows,
  intended for reuse in demonstration-oriented inference pipelines.

### Role in the AstroPredict Pipeline
This notebook serves as a **supporting utility** for:
- Notebook 04 (Real-Time Inference Demonstration)
- Notebook 05 (API Demonstration)

It enables consistent, repeatable demonstrations of model behavior
under different solar activity contexts without claiming real-time
magnetic field availability.

All generated outputs are clearly documented and explicitly labeled
for demonstration purposes only.
